# Astro Agents: quickstart

Verifiable planetary timing data as pandas DataFrames, from the NASA/JPL DE440 ephemeris. Every result is SHA-256 verifiable.
Data for your own analysis, never a trading signal or forecast.

`pip install astro-agents`. Without a token you get 1 free call on the $0.01-0.02 routes; for the finance routes set `ASTRO_CREDITS_TOKEN` (prepaid credits, see `client.packs()`).

In [ ]:
from astro_agents import Client, verify_dataframe
client = Client()          # reads ASTRO_CREDITS_TOKEN
client

## Credit packs (free call)

In [ ]:
client.packs()[["name", "price_usd", "credits", "bonus_pct", "value"]]

## Positions: geocentric vs heliocentric, with declination ($0.01 each)

In [ ]:
geo = client.positions("2024-03-01T00:00Z", coordinates="equatorial")
geo[["sign", "position", "longitude", "declination", "out_of_bounds", "retrograde"]]

In [ ]:
helio = client.positions("2024-03-01T00:00Z", perspective="heliocentric")
helio[["sign", "position", "longitude"]]

## Gann planetary lines as a time series ($0.20 per call, up to 1000 samples each)

In [ ]:
lines = client.planetary_lines(start="2024-01-01", end="2024-12-31", bodies=["jupiter", "saturn"],
                               averages=["mean_of_5"], unwrap=True)
lines.filter(like="jupiter").plot(figsize=(10, 4), title="Jupiter, geocentric vs heliocentric (unwrapped)");

## Event calendar: eclipses and stations ($0.25 per year of events)

In [ ]:
events = client.events("2024-01-01", "2024-12-31", include=["eclipse", "station"])
events[["time", "type", "body", "kind", "eclipse_type", "turns", "position"]]

## Merge onto candles

In [ ]:
import pandas as pd
from astro_agents import merge_events
candles = pd.DataFrame({"open_time": pd.date_range("2024-01-01", "2024-12-31", freq="D", tz="UTC")})
merged = merge_events(candles, events, on="open_time", tolerance="1D")
merged.dropna(subset=["type"]).head()

## Verify: same request, same bytes

In [ ]:
print(verify_dataframe(lines), verify_dataframe(events))      # offline recomputation of result_sha256
lines.attrs["result_sha256"], events.attrs["meta"]["input_sha256"] if isinstance(events.attrs["meta"], dict) else None